# Notebook 01: Explore — Debutanizer Soft Sensor
Stage 1 (load & check) and Stage 2 (exploration) of the debutanizer soft sensor project.

The dataset contains **2394 normalised process samples** from an industrial debutanizer column.
Original engineering units and variable definitions are not publicly disclosed.
The file `debutanizer_data_estimated_real_units.csv` uses assumed scale factors only.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_io import load_training_data, generate_data_summary

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120


## Stage 1: Load and Validate

In [ ]:
df = load_training_data()
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nConfirm 2394 rows:", len(df) == 2394)
print("Confirm values in [0,1]:", df.min().min() >= 0 and df.max().max() <= 1)


In [ ]:
df.describe().round(4)

In [ ]:
summary = generate_data_summary(df)
print("Data summary saved to results/data_summary.md")


## Stage 2: Explore

In [ ]:
# Time-series plot of each variable
fig, axes = plt.subplots(8, 1, figsize=(14, 20), sharex=True)
for i, col in enumerate(df.columns):
    axes[i].plot(df.index, df[col], linewidth=0.8, alpha=0.9,
                 color=plt.cm.tab10.colors[i])
    axes[i].set_ylabel(col, fontsize=10, fontweight="bold")
    axes[i].set_ylim(-0.05, 1.05)
axes[-1].set_xlabel("Row Index (time order)", fontsize=12)
plt.suptitle("All Variables vs. Row Number (Chronological Order)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../figures/timeseries_all_vars.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Histogram of each variable
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, col in enumerate(df.columns):
    ax = axes[i // 4, i % 4]
    ax.hist(df[col], bins=40, color=plt.cm.tab10.colors[i], edgecolor="white", alpha=0.85)
    ax.set_title(col, fontweight="bold")
    ax.set_xlabel("Normalised Value [0, 1]")
    ax.set_ylabel("Count")
plt.suptitle("Histograms of All Variables", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/histograms_all_vars.png", dpi=200)
plt.show()


In [ ]:
# Correlation heatmap
corr = df.corr()
plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.7})
plt.title("Correlation Matrix — All Variables", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/correlation_heatmap.png", dpi=200)
plt.show()

# Flag near-duplicate pairs
print("\nHigh-correlation pairs (|r| > 0.95):")
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        v = corr.iloc[i,j]
        if abs(v) > 0.95:
            print(f"  {corr.columns[i]} vs {corr.columns[j]}: r = {v:.4f}")


In [ ]:
# Scatter plots: each input vs C4 target
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
input_cols = [c for c in df.columns if c != "C4"]
for i, col in enumerate(input_cols):
    ax = axes[i // 4, i % 4]
    ax.scatter(df[col], df["C4"], alpha=0.2, s=5, color=plt.cm.tab10.colors[i])
    r = df[col].corr(df["C4"])
    ax.set_title(f"{col}  (r = {r:.3f})", fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("C4")
# Hide the 8th (empty) subplot
axes[1, 3].set_visible(False)
plt.suptitle("Scatter Plots: Each Input vs C4 Target", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/scatter_inputs_vs_c4.png", dpi=200)
plt.show()


## 5 Plain-English Findings from Exploration

1. **u6 and u7 are near-identical** (correlation r = 0.989). Both are bottom temperatures
   that track the same reboiler process. Their combined predictive credit will be split
   arbitrarily between them in individual feature importance — always read them together.

2. **No missing values or duplicate rows** — the dataset is clean and ready to model as-is.

3. **C4 has a wide operational range** (full 0-1 span) with several distinct operating
   regimes visible as clusters in the time-series — this suggests the column experienced
   significantly different feed conditions or setpoint changes over the measurement period.

4. **Most inputs have low linear correlation with C4** (|r| < 0.25 for u1, u4-u7).
   u3 (reflux flow, r ≈ -0.25) and u2 (pressure, r ≈ -0.21) have the strongest
   pairwise associations, hinting that flow and pressure information will be important features.

5. **The C4 series shows a rising trend near the end of the record**, which is a warning
   sign for temporal drift — the test block (last 20%) includes this drift, and models
   trained on the earlier stable period may show systematic positive bias on the test set.
